<a href="https://colab.research.google.com/github/Tr3vour-svg/Senior-AI-Financial-Analyst/blob/main/Senior_AI_Financial_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

imports

In [ ]:
# System dependencies for PDF processing
!apt-get install -y poppler-utils tesseract-ocr

# Python packages
!pip install -qU langchain langchain-community langchain-openai \
    langchain-pinecone pinecone-client \
    "unstructured[all-docs]" pypdf python-dotenv tiktoken

In [ ]:
!pip install langchain-classic


In [ ]:
import os
from dotenv import load_dotenv

# 1. Document Loading and Parsing
# PyPDFDirectoryLoader is great for bulk, Unstructured is better for complex PDF layouts
from langchain_community.document_loaders import PyPDFDirectoryLoader, UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Embeddings and Vector Storage (Pinecone Specific)
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

# 3. LLM and Retrieval Chain
from langchain_openai import ChatOpenAI

from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

# Load your API Keys from Colab Secrets or a .env file
load_dotenv()

Environmental Configuration

In [ ]:
import os
from google.colab import userdata

def setup_environment():
    """
    Configures API keys for a multi-model RAG project.
    Ensure these keys are added to the Colab 'Secrets' (🔑) tab.
    """

    # 1. Core LLM & Embedding Keys

    os.environ["DEEPSEEK_API_KEY"] = userdata.get('DEEPSEEK_API_KEY')
    os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

    # 2. Vector Database Key
    os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

    # 3. LangChain Tracing (Crucial for monitoring retrieval quality)
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
    os.environ["LANGCHAIN_PROJECT"] = "Big-Tech-RAG-Analysis"

    print("✅ All API keys and LangSmith tracing configured.")

# Execute configuration
setup_environment()

Partitioning Cell

In [ ]:
import os
import pickle
from google.colab import drive
from unstructured.partition.pdf import partition_pdf

# 1. Mount Drive for persistent storage
drive.mount('/content/drive', force_remount=True)

file_paths = [
    "/content/Alphabet_2025.pdf",
    "/content/ASML_2025.pdf",
    "/content/Amazon_2025.pdf",
    "/content/Apple_2025.pdf",
    "/content/Broadcom_2025.pdf",
    "/content/Meta_2025.pdf",
    "/content/Microsoft_2025.pdf",
    "/content/NVIDIA_2025.pdf",
    "/content/TSMC_2025.pdf",
    "/content/Tesla_2025.pdf"
]

CACHE_PATH = "/content/drive/MyDrive/pdf_extraction_cache.pkl"

def partition_only(file_path: str):
    file_name = os.path.basename(file_path)
    print(f"🔍 Starting Vision-Based Partitioning: {file_name}")
    try:
        elements = partition_pdf(
            filename=file_path,
            strategy="hi_res", # Uses model-based layout detection
            infer_table_structure=True, # Critical for 10-K financial data
            extract_image_block_types=["Image"],
            extract_image_block_to_payload=False, # Set to False to keep cache size manageable
            chunking_strategy=None # We will chunk in a separate step
        )
        for element in elements:
            element.metadata.filename = file_name
        print(f"✅ Extracted {len(elements)} elements from {file_name}")
        return elements
    except Exception as e:
        print(f"❌ Error on {file_name}: {e}")
        return []

# --- EXECUTION WITH AUTO-SAVE ---
all_raw_elements = []

# Load existing cache if it exists to pick up where we left off
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        all_raw_elements = pickle.load(f)
    processed_files = {el.metadata.filename for el in all_raw_elements}
    print(f"💾 Found cache. {len(processed_files)} files already processed.")
else:
    processed_files = set()

for path in file_paths:
    file_name = os.path.basename(path)

    if file_name in processed_files:
        print(f"⏭️ Skipping {file_name} (already in cache)")
        continue

    if os.path.exists(path):
        raw_doc_elements = partition_only(path)
        all_raw_elements.extend(raw_doc_elements)

        # Intermediate Save: Save after every successful PDF
        with open(CACHE_PATH, 'wb') as f:
            pickle.dump(all_raw_elements, f)
        print(f"💾 Progress saved to Drive after processing {file_name}")
    else:
        print(f"🚫 File missing: {path}")

print(f"\n🚀 Total Raw Elements in Memory: {len(all_raw_elements)}")

Chunking

In [ ]:
from unstructured.chunking.title import chunk_by_title
from unstructured.documents.elements import CompositeElement, Table
import copy

def create_chunks_by_title(all_raw_elements):
    """
    Expert-level chunking: Preserves table structures, maintains metadata,
    and aggressively merges narrative fragments.
    """
    print("🔨 Creating semantic chunks from extracted elements...")

    # 1. Primary Chunking Logic
    # Using larger max_characters to capture full financial sections
    chunks = chunk_by_title(
        all_raw_elements,
        max_characters=4000,           # Upper limit for context window efficiency
        new_after_n_chars=3200,        # Ideal target for 'full' chunks
        combine_text_under_n_chars=1200, # More aggressive merging to keep concepts together
        overlap=300                    # Balanced overlap for cross-chunk coherence
    )

    print(f"✅ Initial pass: {len(chunks)} chunks")

    # 2. Intelligent Post-Processing (Tiny Chunk Cleanup)
    merged_chunks = []
    skip_next = False

    for i, chunk in enumerate(chunks):
        if skip_next:
            skip_next = False
            continue

        chunk_len = len(str(chunk))

        # If chunk is tiny (<150 chars) and not a Table, try to merge it
        if chunk_len < 150 and i < len(chunks) - 1 and not isinstance(chunk, Table):
            # Capture both content and metadata
            combined_text = str(chunk) + "\n\n" + str(chunks[i + 1])

            # Create a new CompositeElement and merge metadata dicts
            merged = CompositeElement(text=combined_text)

            # Expert Tip: Merge metadata to preserve 'filename', 'page_number', and 'languages'
            merged.metadata = copy.deepcopy(chunk.metadata)
            if hasattr(chunks[i+1], 'metadata'):
                # Update with any unique fields from the second chunk
                merged.metadata.__dict__.update(chunks[i+1].metadata.__dict__)

            merged_chunks.append(merged)
            skip_next = True
            print(f"   🔗 Merged fragment {i} into {i+1} (New len: {len(combined_text)})")
        else:
            merged_chunks.append(chunk)

    # 3. Analytics Dashboard
    chunk_lengths = [len(str(c)) for c in merged_chunks]
    avg_len = sum(chunk_lengths) // len(chunk_lengths)

    print(f"\n📊 Final Chunk Statistics:")
    print(f"   Count: {len(merged_chunks)}")
    print(f"   Avg Length: {avg_len} characters")
    print(f"   Min/Max: {min(chunk_lengths)} / {max(chunk_lengths)}")

    # Check for orphaned tables
    tables = [c for c in merged_chunks if isinstance(c, Table)]
    print(f"   Tables Preserved: {len(tables)}")

    return merged_chunks

# --- EXECUTION ---
if 'all_raw_elements' in globals() and len(all_raw_elements) > 0:
    chunks = create_chunks_by_title(all_raw_elements)
else:
    print("⚠️ Error: 'all_raw_elements' not found. Please run the partitioning cell first.")

The "Random Sample" Strategy

In [ ]:
import random

# Print 3 random chunks to verify the 'Unstructured' quality
if 'chunks' in globals() and chunks: # Check if 'chunks' exists and is not empty
    sample_chunks = random.sample(chunks, 3) # Use 'chunks' instead of 'final_chunks'

    for i, chunk in enumerate(sample_chunks):
        print(f"--- Sample Chunk {i+1} ---")
        print(f"Source: {chunk.metadata.filename}")
        print(f"Type: {type(chunk)}")
        print(f"Content Preview: {str(chunk)[:500]}...") # Just the first 500 chars
        print("-" * 30)
else:
    print("Error: 'chunks' variable not found or is empty. Please ensure the chunking cell ran successfully.")

The "Table Integrity" Check

In [ ]:
# Find the first chunk that looks like a table
table_chunks = [c for c in chunks if "Table" in str(type(c))]

if table_chunks:
    print("📊 Found a Table Chunk! Here is the structure:")
    print(table_chunks[0])
else:
    print("No explicit table chunks found. They might be merged into text.")

Visualizing Distribution

In [ ]:
import pandas as pd

# Quick distribution check
lengths = [len(str(c)) for c in chunks]
print(f"Average Chunk Length: {sum(lengths)/len(lengths):.0f}")
print(f"Max Length: {max(lengths)}")
print(f"Min Length: {min(lengths)}")

# If Min Length is < 100, you might have 'noise' that needs cleaning.

Content Separation & Signature Logic

In [ ]:
import os
import pickle
import hashlib
from google.colab import drive, userdata
from langchain_core.documents import Document

# 1. Configuration & Persistent Storage
drive.mount('/content/drive', force_remount=True)
CACHE_DIR = "/content/drive/MyDrive/rag_cache/"
os.makedirs(CACHE_DIR, exist_ok=True)
PROCESSED_CHUNKS_CACHE = os.path.join(CACHE_DIR, "processed_chunks.pkl")

def get_chunks_signature(chunks):
    """Refined signature: detects content changes even if count remains same"""
    sample_text = "".join([str(c)[:100] for c in chunks[:20]])
    sig_input = f"{len(chunks)}-{sample_text}"
    return hashlib.md5(sig_input.encode()).hexdigest()

def extract_structured_data(element):
    """
    Expert refinement: Extracts data while preserving table-to-text relationships.
    """
    data = {"text": str(element), "table_html": None, "image_b64": None}
    el_type = str(type(element))

    # Prioritize Table Metadata
    if "Table" in el_type:
        data["table_html"] = getattr(element.metadata, 'text_as_html', None)

    # Handle Image Metadata
    elif "Image" in el_type:
        data["image_b64"] = getattr(element.metadata, 'image_base64', None)
        # Use narrative description if available to supplement text
        narrative = getattr(element.metadata, 'text_as_narrative', "")
        if narrative:
            data["text"] += f"\nDescription: {narrative}"

    return data

# Prepare for Cell 2
current_sig = get_chunks_signature(chunks)
print(f"🆔 Chunks Signature: {current_sig}")

AI Enrichment with Partial Progress Saving

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize LLM with optimized settings
llm = ChatOpenAI(
    model='deepseek-chat',
    openai_api_key=userdata.get("DEEPSEEK_API_KEY"),
    openai_api_base='https://api.deepseek.com',
    max_tokens=1024,
    timeout=60 # Prevent hanging on huge tables
)

# 2. Enrichment Logic
def process_with_resilience(chunks, signature):
    processed_docs = []

    # LOAD PARTIAL PROGRESS if exists
    if os.path.exists(PROCESSED_CHUNKS_CACHE):
        with open(PROCESSED_CHUNKS_CACHE, 'rb') as f:
            cached_data = pickle.load(f)
            if cached_data.get('sig') == signature:
                processed_docs = cached_data.get('docs', [])
                print(f"🔄 Resuming from chunk {len(processed_docs)}...")

    start_index = len(processed_docs)

    prompt = ChatPromptTemplate.from_template("""
    You are a senior financial analyst. Create a searchable index for this 10-K data.
    Identify: 1) Entity, 2) Metrics (Revenue/Growth), 3) Key Risks.
    Concise output (max 200 words).

    DATA: {data}
    """)
    chain = prompt | llm

    for i in range(start_index, len(chunks)):
        chunk = chunks[i]
        data = extract_structured_data(chunk)

        # Prepare context for LLM
        context = data["text"]
        if data["table_html"]:
            context = f"TABLE (HTML):\n{data['table_html']}\n\nTEXT:\n{context}"

        try:
            summary = chain.invoke({"data": context[:8000]}).content # Token safety cap
        except Exception as e:
            print(f"⚠️ Chunk {i} failed: {e}. Using fallback.")
            summary = "Summary generation failed for this section."

        # Create the searchable document
        doc = Document(
            page_content=f"[AI SUMMARY]: {summary}\n\n[CONTENT]: {data['text'][:3000]}",
            metadata={
                "source": getattr(chunk.metadata, 'filename', 'Unknown'),
                "page": getattr(chunk.metadata, 'page_number', 'Unknown'),
                "has_table": data["table_html"] is not None,
                "summary": summary
            }
        )
        processed_docs.append(doc)

        # SAVE PROGRESS EVERY 5 CHUNKS
        if (i + 1) % 5 == 0 or (i + 1) == len(chunks):
            with open(PROCESSED_CHUNKS_CACHE, 'wb') as f:
                pickle.dump({'sig': signature, 'docs': processed_docs}, f)
            print(f"✅ Progress Saved: {i + 1}/{len(chunks)}")

    return processed_docs

# Run the pipeline
final_processed_docs = process_with_resilience(chunks, current_sig)

Execution & Caching Cell

In [ ]:
import os
import pickle
import datetime

def load_or_create_processed_chunks(chunks, signature, force_refresh=False):
    """
    Expert refinement: State-aware caching that handles partial completions.
    """
    # 1. Handle Force Refresh
    if force_refresh:
        print("🗑️ Force refresh requested. Clearing cache...")
        if os.path.exists(PROCESSED_CHUNKS_CACHE):
            os.remove(PROCESSED_CHUNKS_CACHE)

    # 2. Check for existing cache (Full or Partial)
    if os.path.exists(PROCESSED_CHUNKS_CACHE):
        with open(PROCESSED_CHUNKS_CACHE, 'rb') as f:
            cache_data = pickle.load(f)

        cached_signature = cache_data.get('sig') or cache_data.get('signature')
        processed_docs = cache_data.get('docs') or cache_data.get('processed_chunks', [])

        # Verify Signature
        if cached_signature == signature:
            if len(processed_docs) >= len(chunks):
                print(f"✅ Full Cache Valid! Loaded {len(processed_docs)} chunks. (LLM costs saved: $0.00)")
                return processed_docs
            else:
                print(f"🔄 Partial Cache Found: {len(processed_docs)}/{len(chunks)} completed. Resuming...")
        else:
            print("⚠️ Signature Mismatch: Input data has changed. Restarting processing...")
            processed_docs = [] # Reset for fresh start
    else:
        print("🔄 No cache found. Starting fresh AI enrichment...")
        processed_docs = []

    # 3. Call the Resilient Processing Logic (from previous cell)
    # This function now handles its own internal saving to PROCESSED_CHUNKS_CACHE
    final_docs = process_with_resilience(chunks, signature)

    return final_docs

# --- MAIN EXECUTION ---

# Resolve which variable holds our data (Handling multiple possible naming conventions)
chunks_to_process = globals().get('chunks') or globals().get('final_chunks')

if chunks_to_process:
    # Get unique ID for the current chunk set
    current_signature = get_chunks_signature(chunks_to_process)

    # Toggle this to True if you want to re-run the LLM summaries
    FORCE_REFRESH = False

    # Execute
    processed_chunks = load_or_create_processed_chunks(
        chunks_to_process,
        current_signature,
        force_refresh=FORCE_REFRESH
    )

    # Final Dashboard
    print("\n" + "="*50)
    print("🚀 RAG READY FOR VECTOR UPLOAD")
    print(f"   Final Doc Count: {len(processed_chunks)}")
    print(f"   Signature: {current_signature}")
    print(f"   Storage: {PROCESSED_CHUNKS_CACHE}")
    print("="*50)

    if processed_chunks:
        print(f"\n📝 Preview of Enriched Metadata (Chunk 0):")
        print(f"   Source: {processed_chunks[0].metadata.get('source')}")
        print(f"   Summary: {processed_chunks[0].metadata.get('summary')[:150]}...")
else:
    print("❌ Critical Error: No chunks found in memory. Please run the Chunking cell.")

Check for original text

In [ ]:
# Check if original_text is accessible
if 'processed_chunks' in globals():
    print(f"📄 Checking first processed chunk:")
    first_chunk = processed_chunks[0]
    print(f"Summary (page_content): {first_chunk.page_content[:300]}...")
    print(f"\nMetadata original_text preview: {first_chunk.metadata.get('original_text', 'Not found')[:300]}...")
    print(f"\nHas tables: {first_chunk.metadata.get('has_tables', False)}")
    print(f"Has images: {first_chunk.metadata.get('has_images', False)}")

Hybrid Vector Store and Embedding Caching Logic

In [ ]:
!pip install pinecone-text

In [ ]:
import os
import time
import pickle
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone_text.sparse import BM25Encoder
from google.colab import userdata

# 1. Configuration
INDEX_NAME = "big-tech-rag-2026"
NAMESPACE = "financial-reports"
EMBEDDING_CACHE_PATH = "/content/drive/MyDrive/rag_cache/embeddings_cache.pkl"

# 2. Initialize Pinecone
pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))

# Delete existing index if it uses 'cosine' (incompatible with hybrid search)
if INDEX_NAME in [idx.name for idx in pc.list_indexes()]:
    existing_index = pc.describe_index(INDEX_NAME)
    if existing_index.metric == 'cosine':
        print(f"🗑️ Deleting cosine-based index: {INDEX_NAME}")
        pc.delete_index(INDEX_NAME)
        print("⏳ Waiting for deletion to complete...")
        time.sleep(15)  # Give Pinecone time to delete

# Create Hybrid-Capable Index with dotproduct metric
if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    print(f"🆕 Creating Hybrid-Capable Serverless Index: {INDEX_NAME}...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # Standard for text-embedding-3-small
        metric='dotproduct',  # Required for hybrid (dense + sparse) search
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    # Wait for index to be ready
    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(1)
    print("✅ Index created successfully!")
else:
    print(f"✅ Index '{INDEX_NAME}' already exists with hybrid-compatible configuration.")

# 3. Initialize Embeddings via OpenRouter
raw_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=userdata.get("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1"
)

# 4. Initialize BM25 Sparse Encoder
bm25_encoder = BM25Encoder()

def get_vector_store(documents, index_name, namespace):
    """
    Expert refinement: Upserts documents to Pinecone with BOTH dense and sparse vectors.
    Manually handles sparse vector creation since PineconeVectorStore.add_documents
    doesn't directly accept sparse_encoder parameter.
    """
    print(f"🚀 Upserting {len(documents)} documents to Pinecone via OpenRouter...")

    # Fit BM25 on your documents to learn vocabulary
    print("🎯 Fitting BM25 Encoder to document vocabulary...")
    bm25_encoder.fit([doc.page_content for doc in documents])

    # Get the Pinecone index
    index = pc.Index(index_name)

    # Process documents in batches (to avoid rate limits)
    batch_size = 100
    total_batches = (len(documents) + batch_size - 1) // batch_size

    for batch_num in range(0, len(documents), batch_size):
        batch_docs = documents[batch_num:batch_num + batch_size]
        current_batch = batch_num // batch_size + 1

        print(f"📦 Processing batch {current_batch}/{total_batches} ({len(batch_docs)} docs)...")

        # Extract texts and metadata
        texts = [doc.page_content for doc in batch_docs]
        metadatas = [doc.metadata for doc in batch_docs]

        # Generate dense embeddings
        dense_embeddings = raw_embeddings.embed_documents(texts)

        # Generate sparse embeddings
        sparse_embeddings = bm25_encoder.encode_documents(texts)

        # Prepare vectors for upsert
        vectors = []
        for i, (text, metadata, dense_vec, sparse_vec) in enumerate(
            zip(texts, metadatas, dense_embeddings, sparse_embeddings)
        ):
            vector_id = f"{namespace}_{batch_num + i}"

            vectors.append({
                "id": vector_id,
                "values": dense_vec,
                "sparse_values": sparse_vec,
                "metadata": {
                    **metadata,
                    "text": text[:1000]  # Store first 1000 chars for preview
                }
            })

        # Upsert to Pinecone
        index.upsert(vectors=vectors, namespace=namespace)
        print(f"✅ Batch {current_batch}/{total_batches} uploaded successfully")

    # Create the vector store object for LangChain compatibility
    vector_store = PineconeVectorStore(
        index_name=index_name,
        embedding=raw_embeddings,
        namespace=namespace
    )

    print("✅ Hybrid vector upload complete (Dense + Sparse embeddings stored).")
    return vector_store

# --- EXECUTION ---
if 'processed_chunks' in globals() and processed_chunks:
    # We check if we need to upload
    index_stats = pc.Index(INDEX_NAME).describe_index_stats()
    vector_count = index_stats.get('total_vector_count', 0)

    if vector_count == 0:
        vector_store = get_vector_store(processed_chunks, INDEX_NAME, NAMESPACE)
    else:
        print(f"ℹ️ Index already contains {vector_count} vectors. Skipping initial upload.")
        # Fit BM25 on existing docs for querying
        bm25_encoder.fit([doc.page_content for doc in processed_chunks])
        vector_store = PineconeVectorStore(
            index_name=INDEX_NAME,
            embedding=raw_embeddings,
            namespace=NAMESPACE
        )
else:
    print("⚠️ No processed chunks found to upload.")

Vector Store Retrieval Cell

In [ ]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from google.colab import userdata
import os

def load_existing_vector_store(index_name, namespace):
    """
    Expert refinement: Connects to an existing Pinecone index without re-uploading.
    Uses OpenRouter as the provider for OpenAI embeddings.
    Automatically detects if index supports hybrid search (sparse vectors).
    """
    # 1. Initialize Pinecone connection
    pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))

    # 2. Check if the index exists in your Pinecone account
    active_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name not in active_indexes:
        print(f"❌ Error: Index '{index_name}' not found. Please run the creation cell first.")
        return None, None  # Return None for both vector_store and bm25_encoder

    # 3. Get index details to check configuration
    index_description = pc.describe_index(index_name)
    index_metric = index_description.metric
    is_hybrid_capable = (index_metric == 'dotproduct')

    print(f"📊 Index Configuration:")
    print(f"   • Name: {index_name}")
    print(f"   • Metric: {index_metric}")
    print(f"   • Hybrid Search Support: {'✅ Yes' if is_hybrid_capable else '❌ No (cosine only)'}")
    print(f"   • Namespace: {namespace}")

    # 4. Initialize the embedding model via OpenRouter
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=userdata.get("OPENROUTER_API_KEY"),
        openai_api_base="https://openrouter.ai/api/v1"
    )

    # 5. Connect to the existing Vector Store
    print(f"\n📡 Connecting to Pinecone index: {index_name}...")
    vector_store = PineconeVectorStore(
        index_name=index_name,
        embedding=embeddings,
        namespace=namespace
    )

    # 6. Sanity Check: Get stats to confirm it's not empty
    stats = pc.Index(index_name).describe_index_stats()
    total_vectors = stats.get('total_vector_count', 0)

    if total_vectors == 0:
        print("⚠️ Warning: Index is empty. You may need to upload documents first.")
        return vector_store, None

    # Check for namespaces
    namespaces = stats.get('namespaces', {})
    if namespace in namespaces:
        namespace_count = namespaces[namespace].get('vector_count', 0)
        print(f"✅ Connection successful!")
        print(f"   • Total vectors in index: {total_vectors}")
        print(f"   • Vectors in '{namespace}': {namespace_count}")
    else:
        print(f"✅ Connection successful!")
        print(f"   • Total vectors in index: {total_vectors}")
        print(f"   • Namespace '{namespace}' found: No (using default or empty)")

    # 7. Initialize BM25 encoder if hybrid search is supported
    bm25_encoder = None
    if is_hybrid_capable:
        print(f"\n🔧 Initializing BM25 encoder for hybrid search...")
        bm25_encoder = BM25Encoder()

        # If we have access to processed_chunks, fit the encoder
        if 'processed_chunks' in globals():
            texts = [doc.page_content for doc in processed_chunks if hasattr(doc, 'page_content')]
            if texts:
                print(f"   • Fitting BM25 on {len(texts)} document chunks...")
                bm25_encoder.fit(texts)
                print(f"   • BM25 encoder ready for keyword matching")
            else:
                print("   • No text content found in processed_chunks")
        else:
            print("   • processed_chunks not available (will use default parameters)")

    return vector_store, bm25_encoder

# --- EXECUTION ---
INDEX_NAME = "big-tech-rag-2026"
NAMESPACE = "financial-reports"

# This variable 'vector_store' will now be used by your RetrievalQA chain
# bm25_encoder will be used for hybrid search (if available)
result = load_existing_vector_store(INDEX_NAME, NAMESPACE)

if result is not None:
    vector_store, bm25_encoder = result

    # Quick verification
    print(f"\n" + "="*60)
    print(f"📋 LOAD SUMMARY")
    print(f"="*60)
    print(f"✓ Vector store: Initialized and ready")
    print(f"✓ Embeddings: OpenAI text-embedding-3-small via OpenRouter")

    if bm25_encoder:
        print(f"✓ Hybrid search: Available (sparse + dense)")
        print(f"✓ Use with: PineconeHybridSearchRetriever")
    else:
        print(f"ℹ️ Hybrid search: Not available (dense only)")
        print(f"✓ Use with: Standard Pinecone retriever or MultiQueryRetriever")

    print(f"="*60)
else:
    print("Failed to load vector store. Please check your index name and API keys.")

The Expert Hybrid Retriever Cell

In [ ]:
import nltk
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone_text.sparse import BM25Encoder
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.prompts import PromptTemplate
from google.colab import userdata

# --- 1. SETUP COMPONENTS ---
nltk.download('punkt', quiet=True)

# Sparse Encoder for Keyword Matching
bm25_encoder = BM25Encoder()
# Fit on your processed chunks to learn the 'Big Tech' vocabulary
print("🎯 Fitting BM25 Encoder to 10-K vocabulary...")
bm25_encoder.fit([doc.page_content for doc in processed_chunks])

# Memory for History Awareness
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    k=3,
    return_messages=True
)

# LLM for Query Expansion (Claude via OpenRouter)
llm_for_expansion = ChatOpenAI(
    model="anthropic/claude-3-haiku",
    temperature=0.2,
    openai_api_key=userdata.get("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1"
)

# --- 2. THE ULTIMATE RETRIEVAL LOGIC ---
def get_ultimate_hybrid_retriever(vector_store, chunks):
    """
    Combines:
    - Sparse Search (BM25) for exact terms/complex words
    - Dense Search (Cosine) for semantic meaning
    - Multi-Query Expansion for total context coverage
    """

    # Initialize the base Hybrid Search (Dense + Sparse)
    # alpha=0.7 balances semantic meaning (0.7) and keyword matching (0.3)
    base_hybrid_retriever = PineconeHybridSearchRetriever(
        embeddings=OpenAIEmbeddings(
            model="text-embedding-3-small",
            openai_api_key=userdata.get("OPENROUTER_API_KEY"),
            openai_api_base="https://openrouter.ai/api/v1"
        ),
        sparse_encoder=bm25_encoder,
        index=pc.Index(INDEX_NAME),
        namespace=NAMESPACE,
        alpha=0.7,
        text_key="text" # Add this line to specify the correct metadata key for content
    )

    # Wrap Hybrid Retriever in Multi-Query Expansion
    # This will generate 3 versions of the query and run Hybrid Search for each
    ultimate_retriever = MultiQueryRetriever.from_llm(
        retriever=base_hybrid_retriever,
        llm=llm_for_expansion
    )

    return ultimate_retriever

# --- 3. STANDALONE QUESTION LOGIC ---
rephrase_prompt = PromptTemplate.from_template("""
    Given the following chat history and a follow-up question, rephrase the
    follow-up into a STANDALONE question containing all necessary context.

    Chat History: {chat_history}
    Follow-up: {question}
    Standalone Question:""")

def get_contextual_query(question):
    chat_history = memory.load_memory_variables({})["chat_history"]
    if not chat_history:
        return question

    chain = rephrase_prompt | llm_for_expansion
    response = chain.invoke({"question": question, "chat_history": chat_history})
    return response.content

# --- EXECUTION ---
ultimate_retriever = get_ultimate_hybrid_retriever(vector_store, processed_chunks)
print("🚀 Hybrid Multi-Query Retriever initialized and ready.")

Dry Run: Retrieval Verification Cell

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Component: Question Rephraser Logic
def get_standalone_question(input_data):
    """Rephrases the user question based on chat history."""
    # If no history, just return the question
    if not input_data.get("chat_history"):
        return input_data["question"]

    # Otherwise, use the LLM to rephrase
    rephrase_chain = rephrase_prompt | llm_for_expansion | StrOutputParser()
    return rephrase_chain.invoke(input_data)

# 2. Component: Simulated Query Execution
def dry_run_retrieval(user_query):
    print(f"👤 Original Query: {user_query}")

    # Step A: Rephrase based on history
    # (Simulating a history where we previously talked about Nvidia)
    history = memory.load_memory_variables({})["chat_history"]
    standalone_q = get_standalone_question({"question": user_query, "chat_history": history})
    print(f"🔍 Standalone Question: {standalone_q}")

    # Step B: Multi-Query Expansion & Retrieval
    # The MQ Retriever handles the expansion internally and returns a unique list of docs
    print(f"📡 Expanding query and hitting Pinecone...")
    retrieved_docs = ultimate_retriever.invoke(standalone_q)

    # Step C: Results Analysis
    print(f"\n✅ Successfully retrieved {len(retrieved_docs)} unique chunks.")
    print("-" * 50)

    for i, doc in enumerate(retrieved_docs[:3]): # Show top 3 for brevity
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        summary = doc.metadata.get('summary', 'No summary available')

        print(f"📄 Result {i+1} | Source: {source} | Page: {page}")
        print(f"💡 AI Summary snippet: {summary[:150]}...")
        print(f"📝 Content Preview: {doc.page_content[:200]}...")
        print("-" * 50)

# --- EXECUTION ---
# Mocking a follow-up question to test history awareness
memory.save_context({"input": "What is Nvidia's main revenue source?"}, {"output": "Nvidia's main revenue comes from its Data Center segment."})

# Try a vague follow-up question
dry_run_retrieval("What was the growth rate for that segment in 2025?")

Reranking cell

In [ ]:
!pip install flashrank

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import FlashrankRerank # Lightweight local choice
# OR: from langchain_cohere import CohereRerank (if using Cohere API)
from flashrank import Ranker

# Fix for PydanticUserError
FlashrankRerank.model_rebuild()

def get_reranked_retriever(base_ultimate_retriever):
    """
    Adds a final precision layer to the Hybrid Multi-Query results.
    """
    # 1. Initialize the Reranker
    # Flashrank is great for Colab because it runs on CPU and is very fast.
    compressor = FlashrankRerank(top_n=10)

    # 2. Create the Compression Retriever
    # This takes the ~30-50 docs from your Hybrid Multi-Query and narrows them to the best 5.
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=base_ultimate_retriever
    )

    return compression_retriever

# --- EXECUTION ---
# This wraps your existing hybrid logic with the new reranker
final_precision_retriever = get_reranked_retriever(ultimate_retriever)
print("🎯 Precision Reranker added to the pipeline.")

The Final Analytical Generation Cell

Reasoning LLM

In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

# 1. Primary Reasoning LLM (GPT-4o)
# This model rivals Claude 3.5 Sonnet in table parsing and logical extraction.
reasoning_llm = ChatOpenAI(
    model="openai/gpt-4o",
    openai_api_key=userdata.get("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0  # Factual precision for financial data
)

# 2. Expansion LLM (GPT-4o-mini)
# Highly cost-effective for multi-query expansion and agentic decomposition.
llm_for_expansion = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.2,
    openai_api_key=userdata.get("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1"
)

print("✅ LLMs updated to OpenAI GPT-4o series via OpenRouter.")

In [ ]:
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def generate_final_answer(query, retrieved_docs):
    """
    Expert Financial AI Analyst Generation Logic.
    Integrates Hybrid-Retrieval documents with Chain-of-Thought reasoning.
    """

    # 1. Pre-process Retrieved Documents into structured context
    text_parts = []
    tables_html = []
    metadata_sources = set()

    for doc in retrieved_docs:
        # Extract source/page for the 'Analyst' to reference
        source_info = f"Source: {doc.metadata.get('source', 'Unknown')} (Page: {doc.metadata.get('page', 'N/A')})"
        metadata_sources.add(source_info)

        # Split original text from summaries for the LLM
        content = doc.page_content
        text_parts.append(f"[{source_info}]\n{content}")

        # If the original chunk had a table, we ensure the LLM sees it
        # (Assuming your metadata or content contains table indicators)
        if doc.metadata.get('has_table') or "TABLE (HTML):" in content:
            # Extract HTML block if present
            table_match = re.search(r'### TABLE \d+ \(HTML\):\n(.*?)(?=\n\n|###|$)', content, re.DOTALL)
            if table_match:
                tables_html.append(table_match.group(1))

    # 2. Financial Table Intelligence
    def extract_table_summary(tables):
        if not tables: return "No structured tables found in these chunks."
        summaries = []
        for i, html in enumerate(tables[:3]):
            # Look for currency patterns in the HTML tags
            vals = re.findall(r'[\$£€]\s?\d+(?:\.\d+)?\s?[MBBillionMillion]*', html)
            if vals:
                summaries.append(f"Table {i+1} Metadata: Contains key values {', '.join(vals[:4])}")
        return "\n".join(summaries)

    table_summary_text = extract_table_summary(tables_html)

    # 3. Enhanced System Prompt
    system_prompt = """You are a world-class Financial AI Analyst (Senior Level).

    YOUR ASSETS:
    - Highly accurate 10-K text chunks.
    - Preserved HTML tables for precise fiscal data.

    TASK:
    Analyze the provided context to answer the user's query.
    Use Chain-of-Thought: First identify the figures, then compare/calculate, then conclude.

    RULES:
    - CITE: You must cite [Source, Page] for every number.
    - TABLES: Use the provided HTML tables as the 'Single Source of Truth' for numbers.
    - COMPARISONS: If asked to compare, provide a side-by-side breakdown.
    - INTEGRITY: If the specific metric (e.g., 'Purchase Obligations') isn't in the context, say so.
    """

    # 4. Logic for Query Type
    query_lower = query.lower()
    is_comparison = any(x in query_lower for x in ["compare", "vs", "versus", "difference"])

    context_block = "\n\n".join(text_parts)[:12000] # Safety limit for context window

    user_prompt_template = """
    QUESTION: {query}

    --- RETRIEVED CONTEXT ---
    {context_block}

    --- TABLE ANALYSIS ---
    {table_info}

    INSTRUCTIONS:
    {instruction_type}

    FINAL ANALYTICAL ANSWER:
    """

    instruction_type = (
        "This is a COMPARISON. Provide a side-by-side analysis and calculate % differences if data allows."
        if is_comparison else
        "Provide a direct answer focused on fiscal accuracy and precise citations."
    )

    # 5. Chain Execution
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt_template)
    ])

    chain = prompt | reasoning_llm | StrOutputParser()

    print(f"🧠 Analyst is processing a {'Comparison' if is_comparison else 'Standard'} query...")

    try:
        response = chain.invoke({
            "query": query,
            "context_block": context_block,
            "table_info": table_summary_text,
            "instruction_type": instruction_type
        })

        # Post-processing confidence header
        if any(num in response for num in ["$", "%", "billion"]):
            response = f"### [CONFIRMED FINANCIAL DATA]\n\n{response}"

        return response
    except Exception as e:
        return f"❌ Analyst Error: {str(e)}"

Enhanced Execution with Agentic Fallbacks

In [ ]:
!pip install nest_asyncio

In [ ]:
import traceback
import time
import asyncio
import nest_asyncio
import re
from typing import List

# Global evaluation buffer
eval_samples = []

# ============================================================================
# SIMPLE DECOMPOSE (Minimal - Just returns original query)
# ============================================================================
def decompose_query(query: str) -> List[str]:
    """NO DECOMPOSITION - Use original query only. Clean and fast."""
    return [query]


# ============================================================================
# SIMPLE GENERATION (Your original working version)
# ============================================================================
def generate_final_answer(query: str, docs: List) -> str:
    """Simple generation - no validation, no structured data"""
    if not docs:
        return "No relevant documents found."

    prompt = f"""
    Based on the following documents, answer the question concisely and accurately.

    Question: {query}

    Documents:
    {chr(10).join([f"Document {i}: {doc.page_content[:1500]}" for i, doc in enumerate(docs[:5])])}

    Answer:
    """

    response = llm.invoke(prompt)
    return response.content


# ============================================================================
# PARALLEL RETRIEVAL (async)
# ============================================================================
async def parallel_retrieval(query_list, retriever):
    """Simple parallel retrieval - no fancy filtering"""
    tasks = [retriever.ainvoke(q) for q in query_list]
    results = await asyncio.gather(*tasks)

    seen = set()
    unique = []
    for sublist in results:
        for doc in sublist:
            key = (doc.metadata.get('source'), doc.metadata.get('page'))
            if key not in seen:
                unique.append(doc)
                seen.add(key)
    return unique


# ============================================================================
# CAPTURE FOR EVALUATION
# ============================================================================
def capture_for_eval(query, answer, docs, ground_truth=None, standalone_query=None):
    """Simple capture - no complex metadata"""
    if not docs:
        print("⚠️ No docs to capture")
        return

    question_for_eval = standalone_query if standalone_query else query

    sample = {
        "question": question_for_eval,
        "answer": answer,
        "contexts": [doc.page_content for doc in docs[:5]],
    }

    if ground_truth:
        sample["ground_truth"] = ground_truth

    eval_samples.append(sample)
    print(f"✅ Sample {len(eval_samples)-1} captured")


# ============================================================================
# MAIN EXECUTION (Your original working version)
# ============================================================================
def execute_rag_query_v2(query, retriever, conversation_history=None):
    """
    SIMPLE WORKING RAG - No complex improvements
    This is the prototype that worked!
    """
    print("\n" + "="*70)
    print(f"🚀 RAG EXECUTION")
    print(f"   Query: {query}")
    print("="*70)

    try:
        # Step 1: Enhance query with conversation history (simple)
        enhanced_query = query
        if conversation_history:
            history_prompt = f"""
            Given this conversation history, rephrase the question to be self-contained.

            History: {conversation_history}
            New Question: {query}

            Rephrased question:
            """
            enhanced_query = llm.invoke(history_prompt).content.strip()
            print(f"📝 Rephrased: {enhanced_query}")

        # Step 2: Decompose (just returns original query)
        sub_queries = decompose_query(enhanced_query)
        print(f"🔍 Using query: {sub_queries[0]}")

        # Step 3: Parallel retrieval
        nest_asyncio.apply()
        loop = asyncio.get_event_loop()

        start_time = time.time()
        docs = loop.run_until_complete(parallel_retrieval(sub_queries, retriever))
        duration = time.time() - start_time

        if not docs:
            print("⚠️ No documents retrieved!")
            return "I couldn't find relevant information to answer your question.", []

        print(f"✅ Retrieved {len(docs)} documents in {duration:.2f}s")

        # Step 4: Generate answer (simple version)
        final_answer = generate_final_answer(enhanced_query, docs)

        # Step 5: Capture for evaluation
        capture_for_eval(query, final_answer, docs, standalone_query=enhanced_query)

        # Step 6: Dashboard
        print("\n📊 METRICS")
        print("="*70)
        print(f"Latency: {duration:.2f}s")
        print(f"Documents: {len(docs)}")

        print("\n💡 ANSWER:")
        print("="*70)
        print(final_answer)
        print("="*70)

        return final_answer, docs

    except Exception as e:
        print(f"\n❌ Error: {e}")
        traceback.print_exc()
        return f"Error: {str(e)}", []


# ============================================================================
# UTILITY
# ============================================================================
def inspect_pinecone_index():
    try:
        stats = pc.Index(INDEX_NAME).describe_index_stats()
        print(f"🔍 INDEX: {INDEX_NAME} | Vectors: {stats['total_vector_count']}")
    except Exception as e:
        print(f"⚠️ Pinecone inspect failed: {e}")


print("="*70)
print("✅ SIMPLE WORKING PIPELINE LOADED")
print("="*70)
print("   - No complex decomposition")
print("   - No step-back queries")
print("   - No alignment node")
print("   - No pruning")
print("   - Just retrieval + generation")
print("="*70)
print("⚡ This is your original prototype that worked!")
print("="*70)

PRODUCTION RUN BLOCK

In [ ]:
import nest_asyncio
from collections import deque

# --- SIMPLE MEMORY BUFFER (No LangGraph Overhead) ---

# Simple conversation memory
conversation_memory = deque(maxlen=10)  # Keep last 10 exchanges

def get_conversation_context():
    """Build context from memory"""
    if not conversation_memory:
        return None
    context_parts = []
    for exchange in conversation_memory:
        context_parts.append(f"User: {exchange['user']}\nAssistant: {exchange['assistant']}")
    return "\n".join(context_parts[-3:])  # Last 3 exchanges

# --- INTERACTIVE SESSION ---

if 'ultimate_retriever' not in globals() and 'final_precision_retriever' not in globals():
    print("❌ No retriever found.")
else:
    active_retriever = final_precision_retriever if 'final_precision_retriever' in globals() else ultimate_retriever

    print("\n" + "="*70)
    print("📈 FINANCIAL ANALYST ENGINE (Simple Memory)")
    print("="*70)
    print("\n💼 Ready! Type 'exit' to quit.\n")

    nest_asyncio.apply()

    while True:
        user_input = input("👤 YOU: ")

        if user_input.lower() in ['exit', 'quit', 'bye']:
            print(f"\n💼 Session ended. Samples: {len(eval_samples)}")
            break

        if not user_input.strip():
            continue

        # Get conversation context from simple memory
        context = get_conversation_context()

        # Execute
        start = time.time()
        answer, docs = execute_rag_query_v2(
            user_input,
            active_retriever,
            conversation_history=context
        )
        elapsed = time.time() - start

        # Store in memory
        conversation_memory.append({"user": user_input, "assistant": answer})

        print(f"\n💼 ANALYST: {answer}")
        print(f"⏱️  {elapsed:.1f}s | 📊 {len(docs)} docs\n")

 RAGAS Evaluation

In [ ]:
!pip install -q ragas datasets

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
import pandas as pd
from langchain_openai import ChatOpenAI

# 1. Initialize OpenRouter as the Judge
# OPTION 5: GPT-4o with higher reasoning (Best for complex evaluations)
openrouter_judge = ChatOpenAI(
    model="openai/gpt-4o",
    api_key=userdata.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    model_kwargs={
        "reasoning_effort": "high"  # Forces deeper analysis
    }
)


# 2. Wrap for Ragas compatibility
evaluator_llm = LangchainLLMWrapper(openrouter_judge)
evaluator_embeddings = LangchainEmbeddingsWrapper(raw_embeddings)  # Use your existing embeddings

# 3. Function to prepare dataset for RAGAS
def prepare_ragas_dataset(samples):
    """Convert eval_samples to RAGAS format with correct column names"""
    ragas_format = []
    for sample in samples:
        ragas_format.append({
            "question": sample.get("question", sample.get("user_input", "")),
            "answer": sample.get("answer", sample.get("response", "")),
            "contexts": sample.get("contexts", sample.get("retrieved_contexts", [])),
        })
    return Dataset.from_list(ragas_format)

# 4. Check if we have data to evaluate
if len(eval_samples) == 0:
    print("❌ No evaluation samples captured. Run the interactive session first!")
else:
    print(f"📊 Evaluating {len(eval_samples)} captured samples...")
    print(f"🔧 Using OpenRouter: openai/gpt-4o")

    # 5. Prepare dataset with correct column names
    eval_dataset = prepare_ragas_dataset(eval_samples)
    print(f"✅ Prepared dataset with columns: {eval_dataset.column_names}")

    # 6. Run RAGAS Evaluation
    print("\n🔄 Running RAGAS evaluation with OpenRouter...")

    try:
        results = evaluate(
            dataset=eval_dataset,
            metrics=[faithfulness, answer_relevancy],
            llm=evaluator_llm,
            embeddings=evaluator_embeddings,
            raise_exceptions=False
        )

        # 7. Display results
        print("\n" + "="*70)
        print("📈 RAGAS EVALUATION RESULTS")
        print("="*70)

        # Convert to DataFrame
        results_df = results.to_pandas()

        # Add original questions for reference
        results_df['question'] = eval_dataset['question']

        # Show individual results
        for idx, row in results_df.iterrows():
            print(f"\n--- Sample {idx} ---")
            print(f"Question: {row['question'][:100]}...")
            faithfulness_score = row['faithfulness'] if pd.notna(row['faithfulness']) else 'N/A'
            answer_relevancy_score = row['answer_relevancy'] if pd.notna(row['answer_relevancy']) else 'N/A'
            print(f"Faithfulness: {faithfulness_score}")
            print(f"Answer Relevancy: {answer_relevancy_score}")

        # 8. Summary statistics
        print("\n" + "="*70)
        print("📊 SUMMARY STATISTICS")
        print("="*70)

        for metric in ['faithfulness', 'answer_relevancy']:
            if metric in results_df.columns:
                valid_scores = results_df[metric].dropna()
                if len(valid_scores) > 0:
                    mean_score = valid_scores.mean()
                    std_score = valid_scores.std()
                    min_score = valid_scores.min()
                    max_score = valid_scores.max()
                    print(f"\n{metric.upper()}:")
                    print(f"  Mean: {mean_score:.3f} (±{std_score:.3f})")
                    print(f"  Range: [{min_score:.3f} - {max_score:.3f}]")
                    print(f"  Valid samples: {len(valid_scores)}/{len(results_df)}")
                else:
                    print(f"\n{metric.upper()}: No valid scores generated")

        # 9. Identify problematic samples
        print("\n" + "="*70)
        print("🔍 LOW-PERFORMING SAMPLES")
        print("="*70)

        for metric in ['faithfulness', 'answer_relevancy']:
            if metric in results_df.columns:
                low_scores = results_df[results_df[metric] < 0.5].dropna(subset=[metric])
                if len(low_scores) > 0:
                    print(f"\n⚠️ {metric.upper()} < 0.5 ({len(low_scores)} samples):")
                    for idx, row in low_scores.iterrows():
                        print(f"  Score {row[metric]:.2f}: {row['question'][:80]}...")
                else:
                    print(f"\n✅ No low scores for {metric.upper()} (all >= 0.5)")

        # 10. Save results
        results_df.to_csv("ragas_evaluation_results.csv", index=False)
        print("\n✅ Results saved to 'ragas_evaluation_results.csv'")

        # 11. Optional: Display performance by sample
        print("\n" + "="*70)
        print("📋 DETAILED SCORES TABLE")
        print("="*70)
        display_df = results_df[['question', 'faithfulness', 'answer_relevancy']].copy()
        display_df['faithfulness'] = display_df['faithfulness'].round(3)
        display_df['answer_relevancy'] = display_df['answer_relevancy'].round(3)
        print(display_df.to_string(max_colwidth=50))

    except Exception as e:
        print(f"\n❌ Evaluation failed: {e}")
        import traceback
        traceback.print_exc()

The Backend Core (main.py)

In [ ]:
!pip install langgraph langchain langchain-openai

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Any, Optional, AsyncGenerator
import asyncio
import time
import json
import traceback
from langchain_core.messages import HumanMessage, AIMessage

# Import your existing functions
# Assuming these are available from your imported modules
# from your_rag_functions import (
#     active_retriever,
#     parallel_retrieval,
#     generate_final_answer,
#     capture_for_eval,
#     eval_samples,
#     llm,
#     nest_asyncio,
#     rag_graph  # Optional, if using LangGraph
# )

app = FastAPI(title="Senior AI Financial Analyst API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    query: str
    thread_id: str = "default_session"

class ChatResponse(BaseModel):
    answer: str
    thread_id: str
    metadata: Dict[str, Any]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def get_conversation_history(thread_id: str) -> Optional[str]:
    """Retrieve conversation history from LangGraph memory"""
    try:
        if 'rag_graph' not in globals() or rag_graph is None:
            return None

        config = {"configurable": {"thread_id": thread_id}}
        # Use sync version for simplicity
        state = rag_graph.get_state(config)

        if state and state.values:
            messages = state.values.get("messages", [])
            if messages:
                # Format last few exchanges
                history_parts = []
                for msg in messages[-6:]:  # Last 6 messages (3 exchanges)
                    if isinstance(msg, HumanMessage):
                        history_parts.append(f"User: {msg.content}")
                    elif isinstance(msg, AIMessage):
                        # Truncate long assistant responses
                        content = msg.content[:300] if len(msg.content) > 300 else msg.content
                        history_parts.append(f"Assistant: {content}")
                if history_parts:
                    return "\n".join(history_parts)
        return None
    except Exception as e:
        print(f"⚠️ Failed to get conversation history: {e}")
        return None


async def perform_retrieval(query: str, retriever) -> List:
    """Perform async retrieval with proper event loop handling"""
    try:
        # nest_asyncio.apply() # Moved to main execution block
        loop = asyncio.get_event_loop()

        # Use simple single query retrieval for streaming endpoint
        docs = await loop.run_until_complete(
            retriever.ainvoke(query)
        )

        # Deduplicate
        seen = set()
        unique_docs = []
        for doc in docs:
            key = (doc.metadata.get('source'), doc.metadata.get('page'))
            if key not in seen:
                unique_docs.append(doc)
                seen.add(key)

        return unique_docs[:10]  # Limit to top 10 for streaming
    except Exception as e:
        print(f"⚠️ Retrieval failed: {e}")
        return []


# ============================================================================
# STREAMING GENERATION FUNCTION
# ============================================================================
async def stream_answer(
    query: str,
    retriever,
    thread_id: str,
    conversation_history: Optional[str] = None
) -> AsyncGenerator[str, None]:
    """
    Stream tokens as they're generated for instant feedback
    """
    start_time = time.time()

    try:
        # Send initial metadata
        yield json.dumps({
            "type": "status",
            "content": "🔍 Analyzing your question...",
            "timestamp": time.time()
        }) + "\n"

        # Step 1: Rephrase with history (if needed)
        enhanced_query = query
        if conversation_history:
            yield json.dumps({
                "type": "status",
                "content": "📝 Understanding conversation context...",
                "timestamp": time.time()
            }) + "\n"

            rephrase_prompt = f"""
            Rephrase this question to be self-contained given the conversation.

            Conversation History:
            {conversation_history}

            User's New Question: {query}

            Rephrased standalone question:
            """

            try:
                response = llm.invoke(rephrase_prompt)
                enhanced_query = response.content.strip()
                # Clean up any artifacts
                enhanced_query = enhanced_query.replace('"', '').strip()
                print(f"📝 Rephrased: {enhanced_query}")
            except Exception as e:
                print(f"⚠️ Rephrasing failed: {e}")
                enhanced_query = query

        # Step 2: Retrieval
        yield json.dumps({
            "type": "status",
            "content": "📡 Searching 10-K documents...",
            "timestamp": time.time()
        }) + "\n"

        # Perform retrieval
        docs = await perform_retrieval(enhanced_query, retriever)

        if not docs:
            yield json.dumps({
                "type": "error",
                "content": "No relevant documents found. Please try rephrasing your question.",
                "timestamp": time.time()
            }) + "\n"
            return

        yield json.dumps({
            "type": "status",
            "content": f"✅ Found {len(docs)} relevant documents. Generating answer...",
            "timestamp": time.time()
        }) + "\n"

        # Step 3: Stream the answer token by token
        yield json.dumps({
            "type": "answer_start",
            "timestamp": time.time()
        }) + "\n"

        # Create streaming prompt
        context = "\n".join([
            f"[Document {i}] {doc.page_content[:1200]}"
            for i, doc in enumerate(docs[:5])
        ])

        prompt = f"""
        You are a financial analyst. Based on the following 10-K document excerpts, answer the question concisely and accurately.

        Question: {enhanced_query}

        Documents:
        {context}

        Answer (be specific, cite sources where possible):
        """

        # Stream tokens from LLM
        full_response = ""
        try:
            # Check if LLM supports streaming
            if hasattr(llm, 'stream'):
                stream = llm.stream(prompt)

                for chunk in stream:
                    if hasattr(chunk, 'content') and chunk.content:
                        token = chunk.content
                        full_response += token
                        yield json.dumps({
                            "type": "token",
                            "content": token,
                            "timestamp": time.time()
                        }) + "\n"
            else:
                # Fallback for non-streaming LLM
                response = llm.invoke(prompt)
                full_response = response.content
                # Send entire response as one token
                yield json.dumps({
                    "type": "token",
                    "content": full_response,
                    "timestamp": time.time()
                }) + "\n"

            # Send completion metadata
            latency = time.time() - start_time
            sources = list(set([doc.metadata.get('source', 'unknown') for doc in docs[:5]]))

            yield json.dumps({
                "type": "complete",
                "metadata": {
                    "latency": round(latency, 2),
                    "num_docs": len(docs),
                    "sources": sources,
                    "answer_length": len(full_response)
                },
                "timestamp": time.time()
            }) + "\n"

            # Capture for evaluation (async, don't block)
            try:
                capture_for_eval(query, full_response, docs)
            except Exception as e:
                print(f"⚠️ Evaluation capture failed: {e}")

        except Exception as e:
            yield json.dumps({
                "type": "error",
                "content": f"Generation error: {str(e)}",
                "timestamp": time.time()
            }) + "\n"

    except Exception as e:
        yield json.dumps({
            "type": "error",
            "content": f"Unexpected error: {str(e)}",
            "timestamp": time.time()
        }) + "\n"
        print(traceback.format_exc())


# ============================================================================
# STREAMING ENDPOINT
# ============================================================================
@app.post("/analyze/stream")
async def analyze_stream(request: ChatRequest):
    """
    Streaming endpoint - sends tokens as they're generated
    """
    # Get retriever from globals
    retriever = None
    if 'active_retriever' in globals():
        retriever = active_retriever
    elif 'final_precision_retriever' in globals():
        retriever = final_precision_retriever
    elif 'ultimate_retriever' in globals():
        retriever = ultimate_retriever
    else:
        raise HTTPException(status_code=500, detail="No retriever available")

    # Get conversation history
    conversation_history = get_conversation_history(request.thread_id)

    return StreamingResponse(
        stream_answer(request.query, retriever, request.thread_id, conversation_history),
        media_type="application/x-ndjson"
    )


# ============================================================================
# NON-STREAMING ENDPOINT (Original)
# ============================================================================
@app.post("/analyze", response_model=ChatResponse)
async def analyze(request: ChatRequest):
    """
    Non-streaming endpoint - returns complete response
    """
    try:
        # Get retriever
        retriever = None
        if 'active_retriever' in globals():
            retriever = active_retriever
        elif 'final_precision_retriever' in globals():
            retriever = final_precision_retriever
        elif 'ultimate_retriever' in globals():
            retriever = ultimate_retriever
        else:
            raise HTTPException(status_code=500, detail="No retriever available")

        # Get conversation history
        conversation_history = get_conversation_history(request.thread_id)

        # Execute your existing RAG function

        start_time = time.time()
        answer, docs = execute_rag_query_v2(
            query=request.query,
            retriever=retriever,
            conversation_history=conversation_history
        )
        latency = time.time() - start_time

        # Extract sources
        sources = list(set([doc.metadata.get('source', 'unknown') for doc in docs[:5]]))

        return ChatResponse(
            answer=answer,
            thread_id=request.thread_id,
            metadata={
                "latency_seconds": round(latency, 2),
                "num_docs": len(docs),
                "sources": sources
            }
        )

    except Exception as e:
        print(f"❌ Error: {e}")
        print(traceback.format_exc())
        raise HTTPException(status_code=500, detail=str(e))


# ============================================================================
# HEALTH CHECK ENDPOINT
# ============================================================================
@app.get("/health")
async def health_check():
    """Health check endpoint"""
    retriever_status = False
    if 'active_retriever' in globals() or 'ultimate_retriever' in globals():
        retriever_status = True

    return {
        "status": "healthy",
        "retriever_ready": retriever_status,
        "streaming_supported": hasattr(llm, 'stream') if 'llm' in globals() else False,
        "samples_captured": len(eval_samples) if 'eval_samples' in globals() else 0
    }


# ============================================================================
# EVALUATION ENDPOINTS
# ============================================================================
@app.get("/evaluation/samples")
async def get_evaluation_samples():
    """Return captured evaluation samples"""
    if 'eval_samples' in globals():
        return {"samples": eval_samples, "count": len(eval_samples)}
    return {"samples": [], "count": 0}


@app.post("/evaluation/reset")
async def reset_evaluation_samples():
    """
    Reset the evaluation buffer
    """
    global eval_samples
    if 'eval_samples' in globals():
        eval_samples = []
        return {"status": "reset", "message": "Evaluation samples cleared"}
    return {"status": "error", "message": "eval_samples not found"}


# ============================================================================
# CONVERSATION MANAGEMENT
# ============================================================================
@app.get("/conversation/{thread_id}")
async def get_conversation(thread_id: str):
    """Get conversation history for a thread"""
    if 'rag_graph' in globals() and rag_graph:
        try:
            config = {"configurable": {"thread_id": thread_id}}
            state = rag_graph.get_state(config)
            if state and state.values:
                messages = state.values.get("messages", [])
                conversation = []
                for msg in messages:
                    if isinstance(msg, HumanMessage):
                        conversation.append({"role": "user", "content": msg.content})
                    elif isinstance(msg, AIMessage):
                        conversation.append({"role": "assistant", "content": msg.content})
                return {"thread_id": thread_id, "conversation": conversation, "length": len(conversation)}
        except Exception as e:
            print(f"⚠️ Failed to get conversation: {e}")

    return {"thread_id": thread_id, "conversation": [], "length": 0}


@app.delete("/conversation/{thread_id}")
async def clear_conversation(thread_id: str):
    """
    Clear conversation history for a thread
    """
    # Note: This depends on your LangGraph implementation
    # For MemorySaver, you may need to implement custom clearing
    return {"status": "cleared", "thread_id": thread_id, "message": "Conversation cleared"}


# ============================================================================
# MAIN ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    import uvicorn
    import nest_asyncio
    nest_asyncio.apply() # Apply nest_asyncio patch globally
    print("="*70)
    print("🚀 Senior Financial Analyst API Server")
    print("="*70)
    print(f"✅ Streaming endpoint: /analyze/stream")
    print(f"✅ Standard endpoint: /analyze")
    print(f"✅ Health check: /health")
    print(f"📍 Server: http://0.0.0.0:8000")
    print(f"📚 API Docs: http://0.0.0.0:8000/docs")
    print("="*70)

    # Use uvicorn.Config and uvicorn.Server to explicitly control event loop handling
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    asyncio.run(server.serve())

The Analyst Dashboard (app.py)

In [ ]:

!pip install streamlit -qq

In [ ]:
import streamlit as st
import requests
import time
import json
import uuid
from datetime import datetime
import pandas as pd


st.set_page_config(
    page_title="Agentic 10-K Financial Analyst",
    page_icon="💼",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for better visual feedback
st.markdown("""
<style>
    .status-box {
        background-color: #f0f2f6;
        padding: 0.5rem 1rem;
        border-radius: 0.5rem;
        margin: 0.5rem 0;
        font-size: 0.9rem;
        color: #666;
    }
    .spinner-text {
        display: inline-block;
        margin-left: 0.5rem;
    }
    .stChatMessage {
        padding: 1rem;
        border-radius: 0.5rem;
        margin-bottom: 1rem;
    }
    .source-tag {
        background-color: #e3f2fd;
        padding: 0.2rem 0.5rem;
        border-radius: 0.3rem;
        font-size: 0.8rem;
        display: inline-block;
        margin: 0.2rem;
    }
</style>
""", unsafe_allow_html=True)

# Initialize session state
if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
if "latency_history" not in st.session_state:
    st.session_state.latency_history = []
if "processing" not in st.session_state:
    st.session_state.processing = False

# ============================================================================
# Sidebar
# ============================================================================
with st.sidebar:
    st.image("https://img.icons8.com/color/96/000000/financial-analyst.png", width=80)
    st.markdown("## 🎛️ Controls")

    # Session info
    st.markdown("### 📋 Session")
    st.info(f"🆔 Session ID: `{st.session_state.thread_id[:8]}...`")

    col1, col2 = st.columns(2)
    with col1:
        if st.button("🆕 New Session", use_container_width=True):
            st.session_state.thread_id = str(uuid.uuid4())
            st.session_state.messages = []
            st.session_state.latency_history = []
            st.rerun()
    with col2:
        if st.button("🗑️ Clear Chat", use_container_width=True):
            st.session_state.messages = []
            st.rerun()

    st.markdown("---")

    # API Configuration
    st.markdown("### 🔌 Connection")
    api_url = st.text_input("API URL", value="http://localhost:8000", help="FastAPI backend URL")

    # Test connection
    if st.button("🔍 Test Connection", use_container_width=True):
        try:
            response = requests.get(f"{api_url}/health", timeout=5)
            if response.status_code == 200:
                st.success("✅ Backend connected!")
            else:
                st.error(f"❌ Error: {response.status_code}")
        except Exception as e:
            st.error(f"❌ Cannot connect: {e}")

    st.markdown("---")

    # Performance metrics
    if st.session_state.latency_history:
        st.markdown("### 📊 Performance")
        avg_latency = sum(st.session_state.latency_history) / len(st.session_state.latency_history)
        st.metric("Avg Response Time", f"{avg_latency:.1f}s")
        st.metric("Total Queries", len(st.session_state.latency_history))

        # Latency chart
        if len(st.session_state.latency_history) > 1:
            latency_df = pd.DataFrame({
                "Query": range(1, len(st.session_state.latency_history) + 1),
                "Latency": st.session_state.latency_history
            })
            st.line_chart(latency_df.set_index("Query"))

# ============================================================================
# Main Chat Interface
# ============================================================================
st.title("💼 Senior AI Financial Analyst")
st.markdown("""
    <div style='background-color: #e8f4f8; padding: 1rem; border-radius: 0.5rem; margin-bottom: 1rem;'>
    🎯 <b>Expert 10-K Analysis</b> – Ask about risk factors, financial metrics, supply chain dependencies,
    and comparative analysis across tech giants.
    </div>
""", unsafe_allow_html=True)

# Quick example queries
with st.expander("🔍 Example Queries", expanded=False):
    col1, col2, col3 = st.columns(3)
    with col1:
        if st.button("📊 Risk Factors", use_container_width=True):
            st.session_state.example_query = "What are the specific 'Risk Factors' Broadcom listed regarding their dependency on a 'limited number of customers'?"
    with col2:
        if st.button("💹 Financial Metrics", use_container_width=True):
            st.session_state.example_query = "What was Microsoft's total 'Property and Equipment' for data centers in FY2025?"
    with col3:
        if st.button("🔗 Supply Chain", use_container_width=True):
            st.session_state.example_query = "Map the cascading supply chain dependencies between ASML, TSMC, and NVIDIA"

# Display chat history
for idx, message in enumerate(st.session_state.messages):
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

        # Show sources for assistant messages
        if message["role"] == "assistant" and "sources" in message and message["sources"]:
            with st.expander(f"📚 Sources ({len(message['sources'])})", expanded=False):
                for source in message["sources"]:
                    st.markdown(f"📄 `{source}`")

# ============================================================================
# Chat Input and Processing
# ============================================================================
if prompt := st.chat_input("Ask about 10-K filings, risk factors, financial comparisons..."):
    # Handle example query
    if "example_query" in st.session_state:
        prompt = st.session_state.example_query
        del st.session_state.example_query

    # Add user message to chat
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Assistant response
    with st.chat_message("assistant"):
        # Create placeholders for different UI elements
        status_placeholder = st.empty()
        response_placeholder = st.empty()

        try:
            start_time = time.time()

            # Show initial status
            status_placeholder.markdown("""
            <div class="status-box">
            🔍 <b>Step 1/4:</b> Analyzing your question...
            </div>
            """, unsafe_allow_html=True)

            # Make API request (your existing endpoint)
            response = requests.post(
                f"{api_url}/analyze",
                json={
                    "query": prompt,
                    "thread_id": st.session_state.thread_id
                },
                timeout=120
            )

            # Update status
            status_placeholder.markdown("""
            <div class="status-box">
            📡 <b>Step 2/4:</b> Searching 10-K documents...
            </div>
            """, unsafe_allow_html=True)

            if response.status_code == 200:
                data = response.json()
                answer = data["answer"]
                sources = data["metadata"].get("sources", [])
                latency = data["metadata"].get("latency_seconds", time.time() - start_time)

                # Update status to generating
                status_placeholder.markdown("""
                <div class="status-box">
                💡 <b>Step 3/4:</b> Generating analysis...
                </div>
                """, unsafe_allow_html=True)

                # Simulate typing effect (visual only)
                full_response = ""
                words = answer.split()
                for i in range(0, len(words), 3):
                    full_response = " ".join(words[:i+3])
                    response_placeholder.markdown(full_response + " ▌")
                    time.sleep(0.01)  # Tiny delay for typing effect

                # Final response without cursor
                response_placeholder.markdown(answer)

                # Update status to complete
                status_placeholder.markdown(f"""
                <div class="status-box" style="background-color: #d4edda; color: #155724;">
                ✅ <b>Step 4/4:</b> Complete! ⏱️ {latency:.1f}s • 📊 {len(sources)} sources
                </div>
                """, unsafe_allow_html=True)

                # Store in session
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": answer,
                    "sources": sources
                })
                st.session_state.latency_history.append(latency)

                # Show sources in expander
                if sources:
                    with st.expander(f"📚 Retrieved Sources ({len(sources)})", expanded=False):
                        for source in sources:
                            st.markdown(f"📄 `{source}`")

            else:
                error_msg = f"❌ API Error: {response.status_code}"
                status_placeholder.markdown(f"""
                <div class="status-box" style="background-color: #f8d7da; color: #721c24;">
                {error_msg}
                </div>
                """, unsafe_allow_html=True)
                response_placeholder.error(error_msg)
                st.session_state.messages.append({"role": "assistant", "content": error_msg})

        except requests.exceptions.Timeout:
            error_msg = "⏰ Request timed out. Please try a simpler question."
            status_placeholder.markdown(f"""
            <div class="status-box" style="background-color: #f8d7da; color: #721c24;">
            {error_msg}
            </div>
            """, unsafe_allow_html=True)
            response_placeholder.error(error_msg)
            st.session_state.messages.append({"role": "assistant", "content": error_msg})

        except Exception as e:
            error_msg = f"❌ Error: {str(e)}"
            status_placeholder.markdown(f"""
            <div class="status-box" style="background-color: #f8d7da; color: #721c24;">
            {error_msg}
            </div>
            """, unsafe_allow_html=True)
            response_placeholder.error(error_msg)
            st.session_state.messages.append({"role": "assistant", "content": error_msg})

        finally:
            # Clear status after a moment (optional)
            time.sleep(2)
            status_placeholder.empty()

# ============================================================================
# Footer
# ============================================================================
st.markdown("---")
st.markdown(
    "<div style='text-align: center; color: gray; font-size: 0.8rem;'>"
    "🔍 Powered by Pinecone | 🤖 GPT-4o | 📊 10-K Filings 2025-2026"
    "</div>",
    unsafe_allow_html=True
)


requirements.txt

In [ ]:
# ============================================================================
# Core Framework & Web Servers
# ============================================================================
fastapi==0.115.0
uvicorn[standard]==0.30.0
streamlit==1.35.0
python-dotenv==1.0.1
pydantic==2.9.0
pydantic-settings==2.4.0

# ============================================================================
# AI & RAG Stack (LangChain Ecosystem)
# ============================================================================
# Core LangChain
langchain==0.3.0
langchain-core==0.3.0
langchain-community==0.3.0
langchain-openai==0.2.0
langchain-classic==0.1.0

# LangGraph for Stateful Applications
langgraph==0.2.0
langgraph-checkpoint==1.0.0
langgraph-checkpoint-sqlite==1.0.0

# ============================================================================
# Vector Database & Embeddings
# ============================================================================
pinecone-client==4.0.0
sentence-transformers==3.0.1
faiss-cpu==1.8.0  # Local vector store fallback
chromadb==0.5.0   # Alternative vector DB

# ============================================================================
# Document Processing
# ============================================================================
unstructured==0.14.0
langchain-unstructured==0.1.0
pypdf==4.2.0
pdfplumber==0.10.3
markdown==3.6
beautifulsoup4==4.12.3

# ============================================================================
# Reranking & Retrieval Optimization
# ============================================================================
flashrank==0.2.0
rank-bm25==0.2.2

# ============================================================================
# Evaluation & Analysis (RAGAS)
# ============================================================================
ragas==0.1.12
datasets==3.0.0
pandas==2.2.2
numpy==1.26.3
scikit-learn==1.5.1
matplotlib==3.9.0
seaborn==0.13.2

# ============================================================================
# API & Networking
# ============================================================================
requests==2.32.0
aiohttp==3.10.0
httpx==0.27.0
nest-asyncio==1.6.0

# ============================================================================
# Utilities & Helpers
# ============================================================================
python-multipart==0.0.9  # For file uploads in FastAPI
tiktoken==0.7.0  # Token counting for OpenAI models
tenacity==8.5.0  # Retry logic for API calls
rich==13.7.1  # Better console output
tqdm==4.66.5  # Progress bars
pyyaml==6.0.2  # YAML config support
loguru==0.7.2  # Better logging
python-json-logger==2.0.7  # JSON structured logging

# ============================================================================
# Notebook / Development Tools
# ============================================================================
jupyter==1.0.0
ipykernel==6.29.4
ipywidgets==8.1.2
nbformat==5.10.4

# ============================================================================
# Monitoring & Tracing (Optional)
# ============================================================================
langsmith==0.1.0  # LangSmith tracing
opentelemetry-api==1.25.0  # For production monitoring
opentelemetry-sdk==1.25.0

# ============================================================================
# Database for Production Memory (Optional)
# ============================================================================
psycopg2-binary==2.9.9  # PostgreSQL for production memory
sqlalchemy==2.0.32  # SQL ORM
alembic==1.13.0  # Database migrations

# ============================================================================
# Deployment & Production
# ============================================================================
gunicorn==22.0.0  # Production WSGI server
redis==5.0.1  # Caching and rate limiting
celery==5.4.0  # Async task queue

docker-compose.yml

In [ ]:
version: '3.8'

services:
  # ============================================================================
  # FastAPI Backend Service
  # ============================================================================
  backend:
    build:
      context: .
      dockerfile: Dockerfile.backend
    image: financial-analyst-backend:latest
    container_name: financial-analyst-backend
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - PINECONE_API_KEY=${PINECONE_API_KEY}
      - PINECONE_ENVIRONMENT=${PINECONE_ENVIRONMENT}
      - PINECONE_INDEX_NAME=${PINECONE_INDEX_NAME}
      - LANGCHAIN_API_KEY=${LANGCHAIN_API_KEY}
      - LANGCHAIN_TRACING_V2=${LANGCHAIN_TRACING_V2:-false}
      - LOG_LEVEL=${LOG_LEVEL:-info}
    volumes:
      - ./checkpoints:/app/checkpoints  # Persistent LangGraph memory
      - ./logs:/app/logs
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s
    restart: unless-stopped
    networks:
      - analyst-network

  # ============================================================================
  # Streamlit Frontend Service
  # ============================================================================
  frontend:
    build:
      context: .
      dockerfile: Dockerfile.frontend
    image: financial-analyst-frontend:latest
    container_name: financial-analyst-frontend
    ports:
      - "8501:8501"
    environment:
      - API_URL=http://backend:8000
    depends_on:
      backend:
        condition: service_healthy
    restart: unless-stopped
    networks:
      - analyst-network

  # ============================================================================
  # PostgreSQL for LangGraph Production Memory (Optional)
  # ============================================================================
  postgres:
    image: postgres:15-alpine
    container_name: analyst-postgres
    environment:
      - POSTGRES_USER=analyst
      - POSTGRES_PASSWORD=${DB_PASSWORD:-changeme}
      - POSTGRES_DB=langgraph_memory
    volumes:
      - postgres_data:/var/lib/postgresql/data
    ports:
      - "5432:5432"
    networks:
      - analyst-network
    restart: unless-stopped
    profiles:
      - production

  # ============================================================================
  # Redis for Caching (Optional)
  # ============================================================================
  redis:
    image: redis:7-alpine
    container_name: analyst-redis
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data
    networks:
      - analyst-network
    restart: unless-stopped
    profiles:
      - production

# ============================================================================
# Networks & Volumes
# ============================================================================
networks:
  analyst-network:
    driver: bridge

volumes:
  postgres_data:
  redis_data:

Dockerfile.backend

In [ ]:
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    curl \
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt && \
    pip install --no-cache-dir psycopg2-binary langgraph-checkpoint-postgres

# Copy application
COPY . .

# Create non-root user
RUN useradd -m -u 1000 appuser && chown -R appuser:appuser /app
USER appuser

HEALTHCHECK --interval=30s --timeout=10s --start-period=40s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000

# Production-grade uvicorn with multiple workers
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4", "--loop", "uvloop", "--http", "httptools"]

Dockerfile.frontend

In [ ]:
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

# Install only frontend requirements (minimal)
RUN pip install --no-cache-dir streamlit==1.35.0 requests==2.32.0 pandas==2.2.2

# Copy frontend app
COPY app.py .

# Create non-root user
RUN useradd -m -u 1000 appuser && chown -R appuser:appuser /app
USER appuser

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0", "--server.headless", "true"]

In [ ]:
# Terminal 1: FastAPI backend
python fastapi_backend.py

# Terminal 2: Streamlit frontend
streamlit run streamlit_frontend.py